# G1 Academy Bonus - Task 5: robot state, services & mode switching (using the wrapper)


## Introduction
Starting today you use the finished `sdk_wrapper.G1` wrapper directly, the same way Task 1 did -- you are no longer rebuilding native `LocoClient`/`RobotStateClient`/DDS calls by hand. This task covers reading robot state, listing/toggling services, and switching FSM modes safely.

**Using Codex/AI for this task:** every method below is a finished, documented method on `sdk_wrapper.G1` -- you are not reconstructing it from DDS. If you get stuck, paste the method's docstring/signature (or the relevant line from `wrapper_cheatsheet.html`) into Codex and ask it to write the cell for you, then read what it produced before you run it against the robot. Knowing *what a call does* and *when it is safe to make it* is the point of this task, not typing it from memory.


In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1

g1 = G1(iface="eth0", domain_id=0)


## Task 1 - Read robot state: `get_lowstate()`, `get_odom()`, `get_battery()`, `get_state()`
- `g1.get_lowstate()` -- a snapshot dict of every motor (`q`, `dq`, `tau_est`, ...) and the IMU, taken from the latest `LowState_` message.
- `g1.get_odom()` -- the robot's current pose/twist from the odometry topic.
- `g1.get_battery()` -- a dict with the battery percentage and related BMS fields.
- `g1.get_state()` -- the composite view: FSM `id`, `mode`, `motion_mode`, `gait`, plus the battery, lowstate, service list, and SLAM info in one call. This is what to reach for when you just want "what mode is the robot in right now" -- there is no separate `get_mode()`; `get_state()["mode"]` / `get_state()["id"]` is it.


In [ ]:
# TODO: call get_lowstate(), get_odom(), get_battery(), and get_state(); print a short summary of each.
raise NotImplementedError("Complete this task section")


## Task 2 - List and toggle services: `get_service()` / `toggle_service()` / `set_service()`
- `g1.get_service()` (no args) -- returns every known service as `{"name", "description", "status", "protected"}`. `g1.get_service("vui_service")` -- returns just that one row (or `None`).
- `g1.toggle_service(name)` -- flips a service based on its *current* reported status.
- `g1.set_service(name, enabled)` -- like `toggle_service`, but sets an explicit on/off instead of flipping. Prefer this one when you know the target state you want (safer than toggling blind).
See `mappings_and_constraints.html` §7 for the full service catalog (`ai_sport`, `basic_service`, `g1_arm_example`, `vui_service`, `unitree_slam`).


In [ ]:
# TODO: list all services, then toggle one (e.g. "vui_service") off and back on, printing the row before/after.
raise NotImplementedError("Complete this task section")


## Task 3 - Switch modes safely: `damp_mode()` / `prepare_mode()` / `walk_mode()` / `run_mode()` / `toggle_dev_mode()`
- `damp_mode()` -- FSM `1`: bounded joint damping, no locomotion. The always-available safe fallback -- call it before an emergency stop or before releasing controller ownership.
- `prepare_mode()` -- FSM `4`: the stand-up/ready pose, the usual step before `walk_mode()`.
- `walk_mode()` -- FSM `500` on this hardware (not `501` -- see the FSM note on the slide: this academy's units run with the waist locked, only `WaistYaw` free).
- `run_mode()` -- FSM `802`.
- `toggle_dev_mode()` -- flips the `ai_sport` service; several Day 3 low-level control calls need dev mode enabled first.
Always go `damp_mode()` → `prepare_mode()` → `walk_mode()` in that order, one step at a time, checking `get_state()` between steps -- never jump straight to `run_mode()`.


In [ ]:
# TODO: step damp_mode() -> prepare_mode() -> walk_mode(), checking get_state() between each step.
raise NotImplementedError("Complete this task section")


## Bonus - a live services dashboard, built with Codex
Prompt Codex with `g1.get_service()`'s return shape and `g1.toggle_service(name)`'s signature and ask it to build a small live view: a polling loop or an `ipywidgets` panel that lists every service with its status and a button to toggle it. This is the same pattern you will use again in Day 3's end-effector teleoperation dashboard.


In [ ]:
# Open-ended: build (or Codex-generate) a minimal service list/toggle dashboard here.


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
